# Objective

The paper ["Matrix Profile I"](https://www.cs.ucr.edu/~eamonn/PID4481997_extend_Matrix%20Profile_I.pdf) uses the [MASS algorithm](https://www.cs.unm.edu/~mueen/FastestSimilaritySearch.html) to compute the distance between a query $Q$ and every subsequence of length $len(Q)$ in $T$. As part of this algorithm, the sliding dot product (sdp) is calculated. The [MASS paper](https://link.springer.com/epdf/10.1007/s10618-024-01005-2?sharing_token=067pAxnaLDvz89q1n_GJt_e4RwlQNchNByi7wbcMAY6HNwOWuMxQSNE3HcKcuL8siHB8L8krJpchQVaGvicUgoegxxV7BWgaU4Y9evg1FU1LGtxlvM9A5UrxrtkqDHyvoOPk_ttVNps-6-LSRf1vuRgFWJI7qrkqWGitjsXfRsw%3D) provided different versions of MASS for computing the sliding dot product. At their core, they all come down to computing one/more convolution(s) via the frequency domain. To understand how convolution comes into the picture and how it is used for computing the sliding dot product, this notebook is created to answer the following questions:

1. **How is the sliding dot product (sdp) related to (linear) convolution?** This can help us discover the relationship between sdp and linear convolution. Convolution is a well-studied area and understanding this relationship allows us to leverage convolution methods to compute the sliding dot product.
2. **How can we compute the convolution faster?** Once we understand the relationship between sdp and convolution, we can focus on learning/using method for faster convolution.
3. **Can we reduce the output size, and hence the computational load without losing the sdp?** Once we know how to get faster convolution, we can try to enhance its performance further by avoiding unnecessary calculation.
4. **What to do if $T$ is very long? Overlap-add method!** This is where we learn a divide-and-conquer algorithm on convolution, a useful method when input $T$ is large

The content of this notebook is based on the book "Discrete-Time Signal Processing" by Alan V. Oppenheim and Ronald W. Schafer. The book is [publicly available in MIT OpenCourseWare](https://ocw.mit.edu/courses/res-6-dtsp-discrete-time-signal-processing/resources/mitres_6-dtsp_s26_thirdedition_pdf/). Most textbooks use $x$ and $h$ notation as the two signals in convolution. However, in this notebook, we would like to use $Q$ and $T$ as those are the name of arrays in the realm of STUMPY. Also, we would like to introduce $Q'$, which is simply the reverse of $Q$. So, if $Q$ has length $m$ (indices $0,\ldots,m-1$), then its reverse $Q'$ is defined by

$$Q'[j] = Q[m-1-j], \quad 0 \le j \le m-1,$$

# 1. How is the sliding dot product(sdp) related to (linear) convolution?

### 1.1 Linear Convolution of Two Finite-Length Sequences $Q'$ and $T$

Consider two finite-length sequences, $Q'$ and $T$, of lengths $m$ and $n$, respectively, where $m < n$. Their linear convolution, $C$, can be computed as follows:

$$ C[idx] = \sum_{i=-\infty}^{\infty}{T[i] \times Q'[idx-i]}$$

where $T[.]$ and $Q'[.]$ are zeros for any index that is outside of their range. The linear convolution can be seen as a flip and slide operation, and at each index $idx$, the output is the sum of element-wise product of overlapping elements. So, as long as there is at least one overlapping element, there can be an output. For instance, at index $idx=0$, there is only one overlapping element between $T$ and $Q'$, and that $T[0]$ and $Q'[0]$. And, $C[0] = T[0]Q'[0]$. Therefore, **the linear convolution is NOT the same as the sliding dot product** since sliding dot product is for cases when the overlap covers the full query and not just one element. 

As stated earlier, $T[i]$ is zero if $i$ is outside of the range $0 \le i \le n-1$. Hence, the boundaries of the sum operation in the previous equation can be revised as follows:

$$ C[idx] = \sum_{i=0}^{n-1}{T[i] \times Q'[idx-i]}$$

Note that $Q'[idx-i]$ can be a non-zero value when $0 \le idx-i \le m-1$, or equivalently $i \le idx \le i+m-1$. So, as the index $i$ changes from $0$ to $n-1$, the interval $i \le idx \le i+m-1$ changes accordingly, and the range of $idx$ becomes the union of these intervals, i.e.

$$\bigcup_{i=0}^{n-1} [\,i,\ i+m-1\,]$$

So:
* min(idx) is coming from the lower bound of interval $[\,i,\ i+m-1\,]$ when $i=0$. This gives: $min(idx)=0$
* max (idx) is coming from the upper bound of interval $[\,i,\ i+m-1\,]$ when $i=n-1$. This gives: $max(idx) = (n-1)+m-1 = n+m-2$

So, when $idx$ changes from $0$ to $n+m-2$, there exist an $i$ such that both $T[i]$ and $Q'[idx-i]$ can be non-zero values. This shows that the linear convolution $C$ can have values at indices $\set{0, 1, ..., n + m - 2}$. 

### 1.2 Sliding dot product (of $Q$ and $T$) as a slice of linear convolution (of $Q'$ and $T$)

Let's check out the values of linear convolution at different indices. For convenience, the linear convolution equation is repeated below:

$$ C[idx] = \sum_{i=0}^{n-1}{T[i] \times Q'[idx-i]}$$

> **NOTE** Let $T_{i}$ denote the sequence $\{T[i], T[i+1], ..., T[i+m-1]\}$

* $idx=0 \implies C[0]=T[0]Q'[0]$
* $idx=1 \implies C[0]=T[0]Q'[1] + T[1]Q'[0]$
* ...
* $\textcolor{red}{idx=m-1 \implies C[m-1]=T[0]Q'[m-1] + T[1]Q'[m-2] + ... + T[m-1]Q'[0] = T_{0}.Q}$
* $\textcolor{red}{idx=m \implies C[m]=T[1]Q'[m-1] + T[2]Q'[m-2] + ... + T[m]Q'[0] = T_{1}.Q}$
* ...
* $\textcolor{red}{idx=n-1 \implies C[n-1]=T[n-m]Q'[(n-1)-(n-m)] + T[n-m+1]Q'[(n-1)-(n-m+1)] + ... + T[n-1]Q'[0] = T_{n-m}.Q}$
* ...
* $idx=n+m-2 \implies C[n+m-2]=T[n-1]Q'[(n+m-2)-(n-1)] = T[n-1]Q'[m-1]$

When $m-1 \le idx \le n-1$, $C[idx]$ actually becomes the dot product between $Q$ (the reverse of $Q'$) and $T_{idx-(m-1)}$. Therefore, the linear convolution of $Q'$ and $T$ contains the sliding dot product between $Q$ and $T$ in the slice in $range(m-1,n)$. 

So, once we calculate the linear convolution between $Q'$ and $T$, we have the sliding dot product between $Q$ and $T$ for free! The time complexity of calculating the linear convolution is $O(nm)$. Can we do better? 

# 2. How can we compute the convolution faster than O(nm)?

### 2.1 Convert linear convolution to circular convolution (Step I)

Circular convolution is defined between two sequences that are both periodic and their periods are the same, say $N$. Their circular convolution is also a periodic sequence, with period $N$. The following equation shows how it can be computed for the period in $range(0, N)$.

$$C_{N}[idx] = \sum_{i=0}^{N-1} \tilde{T}[i] \times \tilde{Q'}[(idx-i)_{N}], \quad 0 \le idx \le N-1$$

where, $C_{N}$ represents one period of convolution output. $\tilde{Q'}$ and $\tilde{T}$ are both periodic sequences, each with period $N$. The index $(idx-i)_{N}$ simply means "$idx-i$ modulo $N$", i.e. $(idx-i) \% N$, and it is always between $0$ and $N-1$.

The time complexity of calculating $C_{N}$ using the equation above is $O(N^{2})$. However, as will be shown later in the next section, the time complexity can be reduced to $O(NlogN)$. However, this means nothing for sliding dot product unless we understand how circular convolution can be used to compute the linear convolution, and how $N$ is related to $n$ and/or $m$. 

We first prove that the linear convolution between $Q'$ and $T$ is equivalent to $C_{N}$ between $\tilde{Q'}$ and $\tilde{T}$ if:

* $N = n + m - 1$
* $\tilde{Q'}_{N}$, i.e. one period of $\tilde{Q'}$, is $Q'$ but with $N-m$ zero padding
* $\tilde{T}_{N}$, i.e. one period of $\tilde{T}$, is $T$ but with $N-n$ zero padding

Recall that the linear convolution equation is:

$$ C[idx] = \sum_{i=0}^{n-1}{T[i] \times Q'[idx-i]}$$

So, to show that $C_{N}$, from circular convolution, gives the same value as the linear convolution, we only need to show that $\tilde{Q'}[(idx-i)_{N}]$ and $Q'[idx-i]$ give the same value when $N=n+m-1$. Let $j$ denote $idx-i$. So, all we need to do is to prove that $\tilde{Q'}[(j)_{N}]$ and $Q'[j]$ give the same value for different values of $j$.

**Case I: $j \ge 0$** <br>

Let's compute the upper bound for $j$. Recall that: 
* $j=idx-i$
* $0 \le idx \le N-1$ (according to range of indices of one period in circular convolution)
* $0 \le i \le n-1$ (according to range of indices of elements in $T$)

Therefore,

$$max(j) = max(idx-i) = max(idx) + max(-i) = max(idx) - min(i) = (N-1) - 0 = N-1$$  

So, $j$ changes from 0 to $N-1$. hence, $j$ modulo $N$ is still $j$. Therefore: $\tilde{Q'}[(j)_{N}]=Q'[j]$. Proof is now complete for this case.

**Case II: $j < 0$** <br>

In this case, $Q'[j]=0$. So, all we need to do is to prove that $\tilde{Q'}[(j)_{N}]$ is zero as well. 

Let's compute the lower bound for $j$. Recall that: 
* $j=idx-i$
* $0 \le idx \le N-1$
* $0 \le i \le n-1$

Therefore,

$$min(j) = min(idx-i) = min(idx) + min(-i) = min(idx) - max(i) = 0 - (n-1)$$

So, in this case, $-(n-1) \le j \le -1$. 

Since $\tilde{Q'}[(j)_{N}] == \tilde{Q'}[(j+N)_{N}]$, we can compute the latter instead of the former. To do that, we first need to figure out the range for $j+N$.

$$ -(n-1) \le j \le -1 \implies -(n-1)+N \le j+N \le -1+N $$

Let's plug the value of $N=n+m-1$ into $-(n-1)+N$:

$$(n+m-1)-(n-1) \le j+N \le N-1 \implies m \le j+N \le N-1$$

Since $j+N$ is between $m$ and $N-1$, its value modulo $N$ becomes $j+N$ again, meaning $\tilde{Q'}[(j)_{N}] == \tilde{Q'}[(j+N)_{N}] = \tilde{Q'}[j+N]$. Since the index $j+N$ is $\ge m$, the value $\tilde{Q'}[j+N]$ becomes 0 as the element resides in the zero-padding part. Proof is now complete for this case.

We just proved that the linear convolution can be computed in the form of circular convolution when the arrays have certain lengths and are zero-padded properly.

### 2.2 Compute Circular Convolution in the Frequency Domain (Step II)

The previous part showed that linear convolution between $Q'$ and $T$ can be computed via circular convolution when both arrays are zero-padded till they reach the length $N=n+m-1$, where $n$ and $m$ are the length of $T$ and $Q'$, respectively. However, the computation via the formula provided in the previous section is not faster. In fact, it has the time complexity of $O(N^2)$! So, why did we go through all that trouble to show that linear convolution can be computed via circular convolution? This is because the circular convolution can be computed more efficiently when it is computed with the help of Fast Fourier Transform. So, instead of computing the circular convolution in time domain, i.e.

$$C_{N}[idx] = \sum_{i=0}^{N-1} \tilde{T}[i] \times \tilde{Q'}[(idx-i)_{N}], \quad 0 \le idx \le N-1$$

we can compute it by taking it to the frequency domain:

$$C_{N} = IFFT\left(
FFT(\tilde{T}_{N})
\cdot
FFT(\tilde{Q'}_{N})
\right)$$

and its time complexity becomes $O(NlogN)$.

So, if I can compute the circular convolution faster, it means that I can calculate the linear convolution faster, and therefore I can obtain the sliding dot product faster than before!

# 3. Can we reduce the output size, and hence the computational load without losing the sdp?

Let's revisit the linear convolution:

$$ C[idx] = \sum_{i=0}^{n-1}{T[i] \times Q'[idx-i]}, \quad 0 \le idx \le n+m-2$$



Recall that the length of output is $n + m - 1$. The sliding dot product between $Q$ and $T$ is in $range(m-1, n)$ though. Therefore, if all we care about is the sliding dot product, the computation can stop at index $n$. In other words, $idx$ (of linear convolution) can be from $0$ to $n-1$. How about the Circular convolution? If we accordingly set $N$ to $n$ instead of $n+m-1$, does the slice in $range(m-1,n)$ still reflect the values of sdp? In other words, we need to show that

$$ C[idx] = \sum_{i=0}^{n-1}{T[i] \times Q'[idx-i]}, \quad m-1 \le idx \le n-1$$

and

$$C_{N=n}[idx] = \sum_{i=0}^{n-1}T[i] \times \tilde{Q'}[(idx-i)_{n}], \quad m-1 \le idx \le n-1$$

give the same result. To prove this, we just need to show that $Q'[idx-i]$ and $\tilde{Q'}[(idx-i)_{n}]$ have the same value when $m-1 \le idx \le n-1$. <br>


**Proof:** Let $j$ denote $idx-i$. We need to check two different cases:

**Case I: $j \ge 0$** <br>

Let's compute the upper bound for $j$. Recall that: 

* $j=idx-i$
* $m-1 \le idx \le n-1$
* $0 \le i \le n-1$

$$ j \le max(j)=max(idx-i)=max(idx)+max(-i)=max(idx) - min(i)=(n-1)-0=n-1$$

Since $0 \le j \le n-1$, $\tilde{Q'}[(j)_{n}]$ is the same as $Q'[j]$. Proof is now complete for this case.

**Case II: $j < 0$** <br>
Note that $Q'[j]=0$ for $j < 0$. So, we need to show $\tilde{Q'}[(j)_{n}]$ becomes zero as well in this case.

Let's compute the lower bound for $j$. Recall that:

* $j=idx-i$
* $m-1 \le idx \le n-1$
* $0 \le i \le n-1$

 
* $\tilde{Q'}[(j)_{n}] = \tilde{Q'}[(j+n)_{n}]$. Note that: 
$$j+n \ge min(j)+n \ge min(idx-i)+n \ge min(idx)+min(-i) + n \ge min(idx) - max(i) + n \ge ((m-1) - (n-1)) + n \ge m$$ 

So, when $j<0$, then: $m \le j+n \le n-1$. Therefore $\tilde{Q'}[(j+n)_{n}]$ simply becomes $\tilde{Q'}[j+n]$, and the value is zero as the index falls into the padded zeros. Therefore, $Q'[j]=\tilde{Q'}[j+n]=0$. Proof is now complete for this case. 

We just showed that the sliding dot product can be computed via circular convolution with period $N=n$. As noted in previous section, this can be computed in $O(nlogn)$ which is faster than our baseline $O(nm)$ unless $m$ is small.

# 4. What to do if $T$ is a long sequence? Use Overlap-add method!

It applies a divide-and-conquer algorithm on convolution. We first need to learn about the "linearity" property in linear convolution and how it allows us to divide the problem into similar problems but with smaller sizes. Then, we learn how we can use circular convolution on those smaller problems, and combine their outputs.

### 4.1 The "Linearity" property in Linear Convolution

Suppose the array $T$ can be written as $T^{(1)} + T^{(2)}$. In other words: $T[i] = T^{(1)}[i] + T^{(2)}[i]$, then:

$$ C[idx] = \sum_{i=-\infty}^{\infty}{T[i] \times Q'[idx-i]} $$

$$ C[idx] = \sum_{i=-\infty}^{\infty}{(T^{(1)}[i] \times Q'[idx-i] + T^{(2)}[i] \times Q'[idx-i])} $$

$$ C[idx] = \sum_{i=-\infty}^{\infty}{T^{(1)}[i] \times Q'[idx-i]} + \sum_{i=-\infty}^{\infty}{T^{(2)}[i] \times Q'[idx-i]}$$

$$ C[idx] = C^{(1)}[idx] + C^{(2)}[idx] $$

This shows that a linear convolution has the property of "linearity". Now, we use this property to show that we can compute the linear convolution between $Q'$ and $T$ by breaking $T$ into smaller parts.

In [ ]:
#WIP